# Layer 2 — Apriori pada Struk Januari dan Februari

Aturan asosiasi hanya ditambang dari penjualan `JUL` bulan Januari dan Februari.
Maret tidak ikut, karena bulan itu menjadi data uji XGBoost.

`min_support = 0.01` hampir tidak menghasilkan pasangan: hanya 6 barang yang lolos.
Pada `0.005` ada 33 barang, tetapi tidak ada pasangan yang mencapai ambang itu.
`min_support = 0.001` memberi 39 pasangan pada 68.269 struk Januari–Februari, jadi ambang ini yang dipakai.
Artinya sebuah pasangan harus muncul di sedikitnya 69 struk.

Item di bawah ambang dibuang sebelum matriks one-hot dibentuk. Identitas item adalah `ITEM`, bukan nama.


In [ ]:
from pathlib import Path

import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

def find_project_dir() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "outputs_ril" / "jul_transactions.csv").exists():
            return candidate
    raise FileNotFoundError("Jalankan Layer 1 terlebih dahulu.")

PROJECT_DIR = find_project_dir()
TX_PATH = PROJECT_DIR / "outputs_ril" / "jul_transactions.csv"
RULES_PATH = PROJECT_DIR / "outputs_ril" / "apriori_rules_ril.csv"

MIN_SUPPORT = 0.001
MIN_CONFIDENCE = 0.1
TRAIN_MONTHS = ["2017-01", "2017-02"]

print("--> [INFO] Berjalan pada data riil. min_support=0.001, min_confidence=0.1, bulan aturan=Januari-Februari.")
print(f"--> [INFO] Folder proyek: {PROJECT_DIR}")


## 1. Keranjang historis

Setiap struk menjadi satu baris. Kolom matriks adalah kode `ITEM` yang support-nya mencapai ambang.


In [ ]:
print("--> [INFO] Menyusun keranjang Januari-Februari dan menyaring ITEM di bawah min_support...")
tx = pd.read_csv(TX_PATH, dtype={"NO_BKT": str, "ITEM": str, "nama_tampil": str})
hist = tx.loc[tx["bulan"].isin(TRAIN_MONTHS), ["NO_BKT", "ITEM", "nama_tampil"]].drop_duplicates()
n_baskets = hist["NO_BKT"].nunique()
support = hist.groupby("ITEM")["NO_BKT"].nunique() / n_baskets
frequent = support[support.ge(MIN_SUPPORT)].index
hist = hist.loc[hist["ITEM"].isin(frequent)]
names = hist.drop_duplicates("ITEM").set_index("ITEM")["nama_tampil"]
print(f"Struk historis: {n_baskets:,} | ITEM lolos support: {len(frequent):,}")

basket = (
    hist.assign(ada=True)
    .pivot_table(index="NO_BKT", columns="ITEM", values="ada", fill_value=False, aggfunc="max")
    .astype(bool)
)
print("--> [INFO] Bentuk matriks keranjang:", basket.shape)


## 2. Aturan asosiasi

Hanya aturan dengan satu barang di tiap sisi yang disimpan, karena Layer 3 memasangkan satu antecedent dengan satu consequent.


In [ ]:
print("--> [INFO] Menjalankan Apriori pada struk Januari-Februari...")
itemsets = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True, low_memory=True)
print("Itemset:", len(itemsets))
rules = association_rules(itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules.loc[rules["antecedents"].map(len).eq(1) & rules["consequents"].map(len).eq(1)].copy()
rules["antecedent"] = rules["antecedents"].map(lambda s: next(iter(s)))
rules["consequent"] = rules["consequents"].map(lambda s: next(iter(s)))
rules["antecedent_name"] = rules["antecedent"].map(names)
rules["consequent_name"] = rules["consequent"].map(names)
rules = rules.sort_values(["lift", "confidence"], ascending=False)
keep = [
    "antecedent", "consequent", "antecedent_name", "consequent_name",
    "support", "antecedent support", "consequent support", "confidence", "lift",
]
rules[keep].to_csv(RULES_PATH, index=False)
print("--> [INFO] Aturan tersimpan:", RULES_PATH)
print("Jumlah aturan:", len(rules))
print(rules[keep].head(15).to_string(index=False))
